# R0 F1 Model Prediction and Forecast

In [1]:
%run ../packages.py
import glob
import fastf1

In [3]:
RAW_PATH = '/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw'
PROCESSED_PATH = "/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/processed"
# PYTENSOR_FLAGS = optimizer = fast_compile

## Create Feature sets

1. rolling feature set of points finishes for each driver / car combination
2. rolling feature set of points finishes for each driver
3. rolling feature set of points finishes for each constructur
4. rolling historical performance for each constructor by race
5. rolling historical performance for each driver  by race

### Rolling feature set of points  finishes for each driver combination

In [7]:
from tqdm import tqdm  # optional but nice for progress
import fastf1
from fastf1 import plotting
import pandas as pd


import fastf1

# ORDER MATTERS — enable_cache MUST come before offline_mode
fastf1.Cache.enable_cache(RAW_PATH)
fastf1.Cache.offline_mode(True)  # now this actually works


class F1DataFetcher:
    def __init__(self, cache_dir=RAW_PATH):
        # Same order here too
        fastf1.Cache.enable_cache(cache_dir)
        fastf1.Cache.offline_mode(True)
        self.cache_dir = cache_dir

    def get_race_results(self, year: int, gp_name: str):
        session = fastf1.get_session(year, gp_name, 'R')
        session.load(laps=True, telemetry=True, weather=True, messages=True)
        results = session.results.copy()
        results['Year'] = year
        results['Race'] = gp_name
        results['Event'] = 'Race'
        results['RaceDate'] = session.date
        return results.reset_index(drop=True)

    def get_qualifying_results(self, year: int, gp_name: str):
        session = fastf1.get_session(year, gp_name, 'Q')
        session.load(laps=True, telemetry=True, weather=True, messages=True)
        results = session.results.copy()
        results['Year'] = year
        results['Race'] = gp_name
        results['Event'] = 'Qualifying'
        results['RaceDate'] = session.date
        return results.reset_index(drop=True)

    def get_all_races(self, year: int):
        schedule = fastf1.get_event_schedule(year)
        races = schedule[schedule['EventFormat'] != 'testing']
        all_results = []

        for _, race in races.iterrows():
            try:
                result = self.get_race_results(year, race['EventName'])
                result_q = self.get_qualifying_results(year, race['EventName'])
                all_results.append(result)
                all_results.append(result_q)
                print(f"  ✅ {race['EventName']}")
            except Exception as e:
                print(f"  ⚠️ Skipping {race['EventName']}: {e}")

        return pd.concat(all_results, ignore_index=True)


# # # Use it
f1 = F1DataFetcher(cache_dir=RAW_PATH)
# # # # season_results_24 = f1.get_all_races(2024)
# # # # season_results_25 = f1.get_all_races(2025)
season_results = []
for i in range(2023, 2026):
    print(f"Fetching data for {i}...")
    season_results.append(f1.get_all_races(i))
season_results = pd.concat(season_results, ignore_index=True)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Fetching data for 2023...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data

  ✅ Bahrain Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timin

...

  ✅ São Paulo Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '63', '12', '16', '55', '6', '27', '44', '31', '87', '14', '22', '10', '30', '43', '23', '5', '18', '4', '81']
core           INFO 	Loading data for Las Vegas Grand Prix - Qualifying [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is ex

  ✅ Las Vegas Grand Prix
  ✅ Qatar Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is ex

  ✅ Abu Dhabi Grand Prix


Event
Race          1378
Qualifying    1339
Name: count, dtype: int64

In [17]:


team_df = season_results[season_results['Event'] == 'Race'].groupby(
    ['TeamId', 'RaceDate', 'Race']).agg({'Points': ['sum', 'mean']}).reset_index()

team_df.columns = [i[0]+"_"+i[1] if i[1] else i[0] for i in team_df.columns]
team_df['year'] = team_df['RaceDate'].dt.year

team_df['Points_cumsum_season_team'] = team_df.groupby(
    ['TeamId', 'year'])['Points_sum'].transform(lambda x: x.cumsum())
team_df['Points_rolling_3race_mean_team'] = team_df.groupby(
    ['TeamId'])['Points_mean'].transform(lambda x: x.rolling(window=3, min_periods=1, closed='left').mean())
team_df['Points_rolling_5race_mean_team'] = team_df.groupby(
    ['TeamId'])['Points_mean'].transform(lambda x: x.rolling(window=5, min_periods=1, closed='left').mean())

team_df['Points_previous_mean_team'] = team_df.groupby(
    ['TeamId'])['Points_mean'].transform(lambda x: x.shift(1))
team_df['Points_previous_mean_race_team'] = team_df.groupby(
    ['TeamId', 'Race'])['Points_mean'].transform(lambda x: x.shift(1)).fillna(team_df['Points_previous_mean_team'])


team_df.fillna(0.0, inplace=True)
team_df[(team_df['TeamId'] == 'red_bull') ]

,TeamId,RaceDate,Race,Points_sum,Points_mean,year,Points_cumsum_season_team,Points_rolling_3race_mean_team,Points_rolling_5race_mean_team,Points_previous_mean_team,Points_previous_mean_race_team
505,red_bull,2023-03-05 15:00:00,Bahrain Grand Prix,43.0,21.5,2023,43.0,0.000000,0.00,0.0,0.0
506,red_bull,2023-03-19 17:00:00,Saudi Arabian Grand Prix,44.0,22.0,2023,87.0,21.500000,21.50,21.5,21.5
507,red_bull,2023-04-02 05:00:00,Australian Grand Prix,36.0,18.0,2023,123.0,21.750000,21.75,22.0,22.0
508,red_bull,2023-04-30 11:00:00,Azerbaijan Grand Prix,43.0,21.5,2023,166.0,20.500000,20.50,18.0,18.0
509,red_bull,2023-05-07 19:30:00,Miami Grand Prix,44.0,22.0,2023,210.0,20.500000,20.75,21.5,21.5
...,...,...,...,...,...,...,...,...,...,...,...
569,red_bull,2025-10-19 19:00:00,United States Grand Prix,31.0,15.5,2025,304.0,12.666667,9.80,9.0,10.5
570,red_bull,2025-10-26 20:00:00,Mexico City Grand Prix,15.0,7.5,2025,319.0,13.666667,12.70,15.5,4.0
571,red_bull,2025-11-09 17:00:00,São Paulo Grand Prix,15.0,7.5,2025,334.0,10.666667,12.20,7.5,13.0
572,red_bull,2025-11-23 04:00:00,Las Vegas Grand Prix,25.0,12.5,2025,359.0,10.166667,11.20,7.5,5.5


In [19]:
driver_df = season_results[season_results['Event'] == 'Race'].groupby(
    ['DriverId', 'RaceDate', 'Race']).agg({'Points': ['mean']}).reset_index()

driver_df.columns = [i[0]+"_"+i[1] if i[1] else i[0]
                     for i in driver_df.columns]
driver_df['year'] = driver_df['RaceDate'].dt.year

driver_df['Points_cumsum_season_driver'] = driver_df.groupby(
    ['DriverId', 'year'])['Points_mean'].transform(lambda x: x.cumsum())
driver_df['Points_rolling_3race_mean_driver'] = driver_df.groupby(
    ['DriverId'])['Points_mean'].transform(lambda x: x.rolling(window=3, min_periods=1, closed='left').mean())
driver_df['Points_rolling_5race_mean_driver'] = driver_df.groupby(
    ['DriverId'])['Points_mean'].transform(lambda x: x.rolling(window=5, min_periods=1, closed='left').mean())

driver_df['Points_previous_mean_driver'] = driver_df.groupby(
    ['DriverId'])['Points_mean'].transform(lambda x: x.shift(1))
driver_df['Points_previous_mean_race_driver'] = driver_df.groupby(
    ['DriverId', 'Race'])['Points_mean'].transform(lambda x: x.shift(1)).fillna(driver_df['Points_previous_mean_driver'])
driver_df.fillna(0.0, inplace=True)
driver_df[driver_df['DriverId'] == 'norris']

,DriverId,RaceDate,Race,Points_mean,year,Points_cumsum_season_driver,Points_rolling_3race_mean_driver,Points_rolling_5race_mean_driver,Points_previous_mean_driver,Points_previous_mean_race_driver
1464,norris,2023-03-04 15:00:00,Bahrain Grand Prix,0.0,2023,0.0,0.0,0.000000,0.0,0.0
1465,norris,2023-03-05 15:00:00,Bahrain Grand Prix,0.0,2023,0.0,0.0,0.000000,0.0,0.0
1466,norris,2023-03-18 17:00:00,Saudi Arabian Grand Prix,0.0,2023,0.0,0.0,0.000000,0.0,0.0
1467,norris,2023-03-19 17:00:00,Saudi Arabian Grand Prix,0.0,2023,0.0,0.0,0.000000,0.0,0.0
1468,norris,2023-04-01 06:00:00,Australian Grand Prix,0.0,2023,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
1595,norris,2025-10-26 20:00:00,Mexico City Grand Prix,25.0,2025,342.0,18.0,16.500000,0.0,0.0
1596,norris,2025-11-08 18:00:00,São Paulo Grand Prix,0.0,2025,0.0,21.5,19.333333,25.0,8.0
1597,norris,2025-11-09 17:00:00,São Paulo Grand Prix,25.0,2025,367.0,25.0,21.500000,0.0,0.0
1598,norris,2025-11-23 04:00:00,Las Vegas Grand Prix,0.0,2025,367.0,25.0,22.666667,25.0,9.0


In [23]:
driver_team_df = season_results[season_results['Event'] == 'Race'].groupby(
    ['DriverId', 'TeamId', 'RaceDate', 'Race']).agg({'Points': ['mean']}).reset_index()

driver_team_df.columns = [i[0]+"_"+i[1] if i[1] else i[0]
                          for i in driver_team_df.columns]
driver_team_df['DriverId_TeamId'] = driver_team_df['DriverId'] + \
    "_" + driver_team_df['TeamId']
driver_team_df['year'] = driver_team_df['RaceDate'].dt.year

driver_team_df['Points_cumsum_season_driver_team'] = driver_team_df.groupby(
    ['DriverId_TeamId', 'year'])['Points_mean'].transform(lambda x: x.cumsum())
driver_team_df['Points_rolling_3race_mean_driver_team'] = driver_team_df.groupby(
    ['DriverId_TeamId'])['Points_mean'].transform(lambda x: x.rolling(window=3, min_periods=1, closed='left').mean())
driver_team_df['Points_rolling_5race_mean_driver_team'] = driver_team_df.groupby(
    ['DriverId_TeamId'])['Points_mean'].transform(lambda x: x.rolling(window=5, min_periods=1, closed='left').mean())
driver_team_df['Points_previous_mean_driver_team'] = driver_team_df.groupby(
    ['DriverId_TeamId'])['Points_mean'].transform(lambda x: x.shift(1))
driver_team_df['Points_previous_mean_race_driver_team'] = driver_team_df.groupby(
    ['DriverId_TeamId', 'Race'])['Points_mean'].transform(lambda x: x.shift(1)).fillna(driver_team_df['Points_previous_mean_driver_team'])
driver_team_df.fillna(0.0, inplace=True)
driver_team_df[driver_team_df['DriverId'] == 'hulkenberg']

,DriverId,TeamId,RaceDate,Race,Points_mean,DriverId_TeamId,year,Points_cumsum_season_driver_team,Points_rolling_3race_mean_driver_team,Points_rolling_5race_mean_driver_team,Points_previous_mean_driver_team,Points_previous_mean_race_driver_team
460,hulkenberg,haas,2023-03-05 15:00:00,Bahrain Grand Prix,0.0,hulkenberg_haas,2023,0.0,0.000000,0.0,0.0,0.0
461,hulkenberg,haas,2023-03-19 17:00:00,Saudi Arabian Grand Prix,0.0,hulkenberg_haas,2023,0.0,0.000000,0.0,0.0,0.0
462,hulkenberg,haas,2023-04-02 05:00:00,Australian Grand Prix,6.0,hulkenberg_haas,2023,6.0,0.000000,0.0,0.0,0.0
463,hulkenberg,haas,2023-04-30 11:00:00,Azerbaijan Grand Prix,0.0,hulkenberg_haas,2023,6.0,2.000000,2.0,6.0,6.0
464,hulkenberg,haas,2023-05-07 19:30:00,Miami Grand Prix,0.0,hulkenberg_haas,2023,6.0,2.000000,1.5,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
524,hulkenberg,sauber,2025-10-19 19:00:00,United States Grand Prix,4.0,hulkenberg_sauber,2025,41.0,0.000000,0.0,0.0,0.0
525,hulkenberg,sauber,2025-10-26 20:00:00,Mexico City Grand Prix,0.0,hulkenberg_sauber,2025,41.0,1.333333,0.8,4.0,4.0
526,hulkenberg,sauber,2025-11-09 17:00:00,São Paulo Grand Prix,2.0,hulkenberg_sauber,2025,43.0,1.333333,0.8,0.0,0.0
527,hulkenberg,sauber,2025-11-23 04:00:00,Las Vegas Grand Prix,6.0,hulkenberg_sauber,2025,49.0,2.000000,1.2,2.0,2.0


In [67]:
tmp1 = season_results[season_results['Event'] == 'Race'][[

    'DriverId', 'TeamId', 'RaceDate', 'Points', 'Race'
]]

tmp2 = tmp1.merge(team_df[['TeamId', 'RaceDate', 'Points_rolling_3race_mean_team', 'Points_rolling_5race_mean_team',
                           'Points_previous_mean_team', 'Points_previous_mean_race_team']],
                  left_on=['TeamId', 'RaceDate'], right_on=['TeamId', 'RaceDate'], how='left')
tmp2 = tmp2.merge(driver_df[['DriverId', 'RaceDate', 'Points_rolling_3race_mean_driver', 'Points_rolling_5race_mean_driver',
                             'Points_previous_mean_driver', 'Points_previous_mean_race_driver']],
                  left_on=['DriverId', 'RaceDate'], right_on=['DriverId', 'RaceDate'], how='left')
final = tmp2.merge(driver_team_df[['DriverId','TeamId', 'RaceDate', 'Points_rolling_3race_mean_driver_team', 'Points_rolling_5race_mean_driver_team',
                                  'Points_previous_mean_driver_team', 'Points_previous_mean_race_driver_team']],
                  left_on=['DriverId','TeamId', 'RaceDate'], right_on=['DriverId','TeamId', 'RaceDate'], how='left')

final.head().T

,0,1,2,3,4
DriverId,max_verstappen,perez,alonso,sainz,hamilton
TeamId,red_bull,red_bull,aston_martin,ferrari,mercedes
RaceDate,2023-03-05 15:00:00,2023-03-05 15:00:00,2023-03-05 15:00:00,2023-03-05 15:00:00,2023-03-05 15:00:00
Points,25.0,18.0,15.0,12.0,10.0
Race,Bahrain Grand Prix,Bahrain Grand Prix,Bahrain Grand Prix,Bahrain Grand Prix,Bahrain Grand Prix
Points_rolling_3race_mean_team,0.0,0.0,0.0,0.0,0.0
Points_rolling_5race_mean_team,0.0,0.0,0.0,0.0,0.0
Points_previous_mean_team,0.0,0.0,0.0,0.0,0.0
Points_previous_mean_race_team,0.0,0.0,0.0,0.0,0.0
Points_rolling_3race_mean_driver,0.0,0.0,0.0,0.0,0.0


In [ ]:
# add unique id to driver team piaring
season_results_24['driver_team_id'] = season_results_24['DriverId'] + \
    "-" + season_results_24['TeamId']
season_results_25['driver_team_id'] = season_results_25['DriverId'] + \
    "-" + season_results_25['TeamId']

# create priors for 24 season to basis 25 season
driver_team_points_24 = season_results_24.groupby('driver_team_id')[
    'Points'].mean().to_dict()

driver_points_24 = season_results_24.groupby(
    'DriverId')['Points'].mean().to_dict()

team_24 = season_results_24.groupby(
    'TeamId')['Points'].mean().to_dict()


# create priors for 24 season to basis 25 season
driver_team_points_25 = season_results_25.groupby('driver_team_id')[
    'Points'].mean().to_dict()

driver_points_25 = season_results_25.groupby(
    'DriverId')['Points'].mean().to_dict()

team_25 = season_results_25.groupby(
    'TeamId')['Points'].mean().to_dict()

# What existed in 2024
known_driver_team_combos = set(season_results_24['driver_team_id'].unique())
known_drivers = set(season_results_24['DriverId'].unique())
known_teams = set(season_results_24['TeamId'].unique())

# Flag anything in 2025 that wasn't in 2024
season_results_25['is_new_driver_team'] = (
    ~season_results_25['driver_team_id'].isin(known_driver_team_combos)
).astype(int)

season_results_25['is_new_driver'] = (
    ~season_results_25['DriverId'].isin(known_drivers)
).astype(int)

season_results_25['is_new_team'] = (
    ~season_results_25['TeamId'].isin(known_teams)
).astype(int)

season_results_24['driver_team_points_prev_1'] = season_results_24.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(1))
season_results_24['driver_team_points_prev_2'] = season_results_24.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(2))
season_results_24['driver_team_points_prev_3'] = season_results_24.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(3))

pre_season_seed = season_results_24.groupby('driver_team_id').tail(3)
pre_season_seed['previous_season'] = 1

current_season = pd.concat(
    [pre_season_seed, season_results_25], ignore_index=True)
current_season['previous_season'] = current_season['previous_season'].fillna(0)
current_season.sort_values('RaceDate', inplace=True)

current_season['driver_team_points_prev_1'] = current_season.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(1))
current_season['driver_team_points_prev_2'] = current_season.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(2))
current_season['driver_team_points_prev_3'] = current_season.groupby('driver_team_id')['Points'].transform(
    lambda x: x.shift(3))


# # Step 3: Merge back onto your main DataFrame
# season_results_25 = season_results_25.merge(
#     team_race_points[['TeamId', 'RaceDate', 'team_rolling_3_mean']],
#     on=['TeamId', 'RaceDate'],
#     how='left'
# )

# team_rolling_test = current_season.groupby('TeamId')['Points'].apply(
#     lambda x: x.rolling(3, min_periods=0).mean()).reset_index(name='team_rolling_3_mean')
# team_rolling_test[team_rolling_test['TeamId'] == 'mclaren']

team_rolling_test = current_season[['Points', 'TeamId', 'RaceDate', 'Race']
                                   ].groupby(["TeamId", "Race", "RaceDate"]).mean().reset_index().sort_values(["RaceDate", "TeamId"]).reset_index(drop=True)

team_rolling_test[team_rolling_test['TeamId'] == 'mclaren']

,TeamId,Race,RaceDate,Points
13,mclaren,Mexico City Grand Prix,2024-10-27 20:00:00,11.0
23,mclaren,Las Vegas Grand Prix,2024-11-24 06:00:00,7.5
33,mclaren,Abu Dhabi Grand Prix,2024-12-08 13:00:00,13.0
43,mclaren,Australian Grand Prix,2025-03-16 04:00:00,13.5
53,mclaren,Japanese Grand Prix,2025-04-06 05:00:00,16.5
63,mclaren,Bahrain Grand Prix,2025-04-13 15:00:00,20.0
73,mclaren,Saudi Arabian Grand Prix,2025-04-20 17:00:00,18.5
83,mclaren,Emilia Romagna Grand Prix,2025-05-18 13:00:00,16.5
93,mclaren,Monaco Grand Prix,2025-05-25 13:00:00,20.0
103,mclaren,Spanish Grand Prix,2025-06-01 13:00:00,21.5


In [230]:
# Step 1: Get total team points per race (both drivers combined)
team_race_points = (
    current_season
    .groupby(['TeamId', 'RaceDate'])['Points']
    .sum()
    .reset_index()
    .rename(columns={'Points': 'team_race_total'})
    .sort_values('RaceDate')
)

# Step 2: Shift first so current race is excluded, then roll
team_race_points['team_rolling_3_mean'] = (
    team_race_points
    .groupby('TeamId')['team_race_total']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# Step 3: Merge back
current_season = current_season.merge(
    team_race_points[['TeamId', 'RaceDate', 'team_rolling_3_mean']],
    on=['TeamId', 'RaceDate'],
    how='left'
)

# Verify both drivers on same team share identical value
print("=== Teammates should match ===")
print(current_season[
    (current_season['TeamId'] == 'mclaren') &
    (current_season['previous_season'] == 0)]
    [['RaceDate', 'Race', 'DriverId', 'Points', 'team_rolling_3_mean']]
    .sort_values('RaceDate')
    .head(10)
)

=== Teammates should match ===
               RaceDate                       Race DriverId  Points  \
87  2025-03-16 04:00:00      Australian Grand Prix   norris    25.0   
88  2025-03-16 04:00:00      Australian Grand Prix  piastri     2.0   
101 2025-04-06 05:00:00        Japanese Grand Prix   norris    18.0   
102 2025-04-06 05:00:00        Japanese Grand Prix  piastri    15.0   
120 2025-04-13 15:00:00         Bahrain Grand Prix  piastri    25.0   
122 2025-04-13 15:00:00         Bahrain Grand Prix   norris    15.0   
142 2025-04-20 17:00:00   Saudi Arabian Grand Prix  piastri    25.0   
148 2025-04-20 17:00:00   Saudi Arabian Grand Prix   norris    12.0   
162 2025-05-18 13:00:00  Emilia Romagna Grand Prix   norris    18.0   
163 2025-05-18 13:00:00  Emilia Romagna Grand Prix  piastri    15.0   

     team_rolling_3_mean  
87             21.000000  
88             21.000000  
101            22.666667  
102            22.666667  
120            28.666667  
122            28.666667 

In [241]:
# Merge the novelty flags onto current_season so we can use them for filling

# For 2024 seed rows, these flags don't apply — fill with 0
current_season[['is_new_driver', 'is_new_team', 'is_new_driver_team']] = (
    current_season[['is_new_driver', 'is_new_team', 'is_new_driver_team']]
    .fillna(0).astype(int)
)


def get_seed_value(row, team_means, driver_team_means):
    """
    Returns the appropriate seed value for NaN lag features.
    Only called for rows where lags are NaN.
    """
    if row['is_new_driver'] == 1 or row['is_new_team'] == 1:
        return 0.0
    if row['is_new_driver_team'] == 1:
        return team_means.get(row['TeamId'], 0.0)
    # Known combo — should have been seeded by pre_season rows
    # If still NaN it means they had fewer than 3 races in 2024
    return driver_team_means.get(row['driver_team_id'], 0.0)


lag_cols = ['driver_team_points_prev_1',
            'driver_team_points_prev_2',
            'driver_team_points_prev_3']

# Only fill NaNs in the 2025 rows, leave 2024 seed rows alone
mask_25 = current_season['previous_season'] == 0
for col in lag_cols:
    nan_mask = mask_25 & current_season[col].isna()
    if nan_mask.any():
        current_season.loc[nan_mask, col] = current_season[nan_mask].apply(
            lambda row: get_seed_value(row, team_24, driver_team_points_24),
            axis=1
        )


current_season.head().T

,0,1,2,3,4
DriverNumber,38,2,2,2,3
BroadcastName,O BEARMAN,L SARGEANT,L SARGEANT,L SARGEANT,D RICCIARDO
Abbreviation,BEA,SAR,SAR,SAR,RIC
DriverId,bearman,sargeant,sargeant,sargeant,ricciardo
TeamName,Ferrari,Williams,Williams,Williams,RB
TeamColor,e8002d,64C4FF,64C4FF,64C4FF,6692FF
TeamId,ferrari,williams,williams,williams,rb
FirstName,Oliver,Logan,Logan,Logan,Daniel
LastName,Bearman,Sargeant,Sargeant,Sargeant,Ricciardo
FullName,Oliver Bearman,Logan Sargeant,Logan Sargeant,Logan Sargeant,Daniel Ricciardo


In [111]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

features = ['Points_rolling_3race_mean_team', 'Points_rolling_5race_mean_team',
       'Points_previous_mean_team', 'Points_previous_mean_race_team',
       'Points_rolling_3race_mean_driver', 'Points_rolling_5race_mean_driver',
       'Points_previous_mean_driver', 'Points_previous_mean_race_driver',
       'Points_rolling_3race_mean_driver_team',
       'Points_rolling_5race_mean_driver_team',
       'Points_previous_mean_driver_team',
       'Points_previous_mean_race_driver_team']
target = 'Points'


# Get ordered list of 2025 races only
race_season_list = (
    final[['Race', 'RaceDate']].drop_duplicates().sort_values('RaceDate').reset_index(drop = True))

ts_split = TimeSeriesSplit(n_splits=len(race_season_list) - 1)
results = []

current_season = final.copy()

for i, (train_index, test_index) in enumerate(ts_split.split(race_season_list)):

    # Convert numpy indices to race names — wrap in list for isin()
    train_race_names = race_season_list.iloc[train_index]
    test_race_names = race_season_list.iloc[test_index]
    train_index
    train_races = current_season[
        (current_season['RaceDate']<=train_race_names['RaceDate'].max())
    ]
    test_races = current_season[
        (current_season['Race'] == test_race_names['Race'].values[0]) &
    (current_season['RaceDate'] == test_race_names['RaceDate'].values[0])

    ]

    X_train = train_races[features]
    y_train = train_races[target]
    X_test = test_races[features]
    y_test = test_races[target]

    # Skip if not enough training data
    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    preds = [int(i) for i in preds]
    rmse = mean_squared_error(y_test, preds, squared=False)

    results.append({
        'fold':        i,
        'train_races': len(train_race_names),
        'test_race':   test_race_names['Race'].values[0],
        'RaceDate': test_race_names['RaceDate'].values[0],
        'rmse':        rmse,
        'n_test_rows': len(y_test)
    })

    print(f"Fold {i:2d} | Test: {test_race_names['Race'].values[0]:<35} | RMSE: {rmse:.3f}")

# Summary
results_df = pd.DataFrame(results)
print(f"\nMean RMSE across folds: {results_df['rmse'].mean():.3f}")

Fold  0 | Test: Saudi Arabian Grand Prix            | RMSE: 7.266
Fold  1 | Test: Australian Grand Prix               | RMSE: 4.583
Fold  2 | Test: Azerbaijan Grand Prix               | RMSE: 6.201
Fold  3 | Test: Miami Grand Prix                    | RMSE: 4.025
Fold  4 | Test: Monaco Grand Prix                   | RMSE: 7.249
Fold  5 | Test: Spanish Grand Prix                  | RMSE: 5.431
Fold  6 | Test: Canadian Grand Prix                 | RMSE: 4.129
Fold  7 | Test: Austrian Grand Prix                 | RMSE: 6.058
Fold  8 | Test: British Grand Prix                  | RMSE: 4.370
Fold  9 | Test: Hungarian Grand Prix                | RMSE: 5.320
Fold 10 | Test: Belgian Grand Prix                  | RMSE: 4.450
Fold 11 | Test: Dutch Grand Prix                    | RMSE: 4.733
Fold 12 | Test: Italian Grand Prix                  | RMSE: 4.422
Fold 13 | Test: Singapore Grand Prix                | RMSE: 7.662
Fold 14 | Test: Japanese Grand Prix                 | RMSE: 5.104
Fold 15 | 

In [119]:
pd.DataFrame(model.feature_importances_.reshape(1,-1),columns = features).T

,0
Points_rolling_3race_mean_team,0.031080
Points_rolling_5race_mean_team,0.066940
Points_previous_mean_team,0.036737
Points_previous_mean_race_team,0.036591
Points_rolling_3race_mean_driver,0.036302
Points_rolling_5race_mean_driver,0.048611
Points_previous_mean_driver,0.038656
Points_previous_mean_race_driver,0.028427
Points_rolling_3race_mean_driver_team,0.070549
Points_rolling_5race_mean_driver_team,0.470008


In [4]:
from collections import deque
import pandas as pd

# Ensure your DataFrame is sorted chronologically within each group
# season_results = season_results.sort_values(['Abbreviation', 'TeamName', 'RaceDate'])


def rolling_list(series, window=5, include_current=True):
    """
    Returns a list of the last `window` values for each element in `series`.
    If include_current=False, excludes the current row's value.
    """
    d = deque(maxlen=window)
    out = []
    for v in series:
        # append a copy of the current rolling window
        if include_current:
            d.append(v)
            out.append(list(d))
        else:
            out.append(list(d))
            d.append(v)
    return pd.Series(out, index=series.index)


# Apply per driver-constructor
season_results['Points_last5'] = (
    season_results
    .groupby(['Abbreviation', 'TeamName'])['Points']
    .apply(rolling_list, window=5, include_current=False)
    .reset_index(level=[0, 1], drop=True)
)

In [16]:
# # f1_bayes_poisson.py
# # import pymc as pm
# # import pytensor.tensor as pt
# # import arviz as az


# class BayesianF1PointsModelPyMC3:
#     """
#     Hierarchical Poisson (optionally Zero-Inflated) for F1 points at the driver–constructor pair level.
#     Fit once on history, then reuse posterior for next-race and season simulations.
#     """

#     def __init__(self,
#                  num_features=("form5", "qualy_pos",
#                                "track_speed_idx", "weather_rain_prob"),
#                  cat_ids=("pair_id", "constructor_id"),
#                  zero_inflated=False,
#                  draws=2000,
#                  tune=2000,
#                  chains=4,
#                  target_accept=0.9,
#                  random_seed=42):
#         self.num_features = list(num_features)
#         self.cat_ids = list(cat_ids)  # expected: ["pair_id", "constructor_id"]
#         self.zero_inflated = zero_inflated
#         self.draws = draws
#         self.tune = tune
#         self.chains = chains
#         self.target_accept = target_accept
#         self.random_seed = random_seed

#         # learned during fit
#         self.model_ = None
#         self.trace_ = None
#         self.train_means_ = {}
#         self.train_stds_ = {}
#         self.n_pairs_ = None
#         self.n_cons_ = None

#         # PM Data containers
#         self._X_data = None
#         self._y_obs = None
#         self._pair_idx = None
#         self._cons_idx = None

#     # ---------- utilities ----------
#     @staticmethod
#     def _as_int_index(series):
#         # expects already integer ids; factorize strings if needed before calling fit
#         return series.astype("int64").values

#     def _fit_standardizer(self, df):
#         for c in self.num_features:
#             x = df[c].astype(float).values
#             m = x.mean()
#             s = x.std() + 1e-8
#             self.train_means_[c] = m
#             self.train_stds_[c] = s

#     def _transform(self, df):
#         # returns X (N,P) standardized, in the order of self.num_features
#         cols = []
#         for c in self.num_features:
#             x = df[c].astype(float).values
#             m = self.train_means_[c]
#             s = self.train_stds_[c]
#             cols.append((x - m) / s)
#         return np.vstack(cols).T.astype(float)

#     # ---------- public API ----------
#     def fit(self, hist_df):
#         """
#         hist_df columns:
#           target: 'points' (int)
#           ids: 'pair_id', 'constructor_id' (int-coded)
#           numeric features: any in self.num_features (float)
#         """
#         required = {"points", *self.cat_ids, *self.num_features}
#         missing = required - set(hist_df.columns)
#         if missing:
#             raise ValueError(f"Missing required columns: {missing}")

#         # id extents
#         self.n_pairs_ = int(hist_df["pair_id"].max()) + 1
#         self.n_cons_ = int(hist_df["constructor_id"].max()) + 1

#         # standardization fit
#         self._fit_standardizer(hist_df)
#         X = self._transform(hist_df)
#         y = hist_df["points"].astype("int64").values
#         pair_idx = self._as_int_index(hist_df["pair_id"])
#         cons_idx = self._as_int_index(hist_df["constructor_id"])

#         with pm.Model() as model:
#             # data holders for future set_data swaps
#             X_data = pm.Data("X_data", X)
#             y_obs = pm.Data("y_obs", y)
#             pair_data = pm.Data("pair_idx", pair_idx)
#             cons_data = pm.Data("cons_idx", cons_idx)

#             # priors
#             alpha = pm.Normal("alpha", 0.0, 1.5)
#             sigma_pair = pm.HalfNormal("sigma_pair", 1.0)
#             sigma_cons = pm.HalfNormal("sigma_cons", 1.0)
#             pair_re = pm.Normal("pair_re", 0.0, sigma_pair,
#                                 shape=self.n_pairs_)
#             cons_re = pm.Normal("cons_re", 0.0, sigma_cons, shape=self.n_cons_)
#             beta = pm.Normal("beta", 0.0, 1.0, shape=X.shape[1])

#             eta = (alpha + pair_re[pair_data] +
#                    cons_re[cons_data] + pt.dot(X_data, beta))
#             lam = pm.Deterministic("lambda", pt.exp(eta))

#             if not self.zero_inflated:
#                 y_like = pm.Poisson("y_like", mu=lam, observed=y_obs)
#             else:
#                 # simple zero-inflation with intercept-only (extend with features if desired)
#                 z_beta0 = pm.Normal("z_beta0", 0.0, 1.0)
#                 psi = pm.Deterministic("psi", pm.math.sigmoid(z_beta0))
#                 y_like = pm.ZeroInflatedPoisson(
#                     "y_like", psi=psi, theta=lam, observed=y_obs)

#             trace = pm.sample(draws=self.draws, tune=self.tune, chains=self.chains,
#                               target_accept=self.target_accept, random_seed=self.random_seed,
#                               return_inferencedata=True)

#         # cache
#         self.model_ = model
#         self.trace_ = trace
#         self._X_data, self._y_obs = X_data, y_obs
#         self._pair_idx, self._cons_idx = pair_data, cons_data
#         return self

#     def predict_next_race(self, race_df, return_draws=True):
#         """
#         race_df: rows = all driver–constructor pairs in the upcoming race.
#         Must include self.cat_ids + self.num_features (numeric features already engineered).
#         Returns (mu_mean, pred_draws) where:
#           - mu_mean: expected points per row (posterior predictive mean of Poisson mu)
#           - pred_draws: posterior predictive integer draws (n_draws, M), if return_draws=True
#         """
#         if self.model_ is None:
#             raise RuntimeError("Call fit() first.")

#         # transform features with training stats
#         Xr = self._transform(race_df)
#         pair_r = self._as_int_index(race_df["pair_id"])
#         cons_r = self._as_int_index(race_df["constructor_id"])

#         with self.model_:
#             pm.set_data({
#                 "X_data": Xr,
#                 "pair_idx": pair_r,
#                 "cons_idx": cons_r,
#                 "y_obs": np.zeros(len(race_df), dtype="int64")  # dummy
#             })
#             ppc = pm.sample_posterior_predictive(
#                 self.trace_, var_names=["y_like", "lambda"])
#         # shapes:
#         #   ppc["lambda"]: (draws, M)
#         mu = ppc["lambda"].mean(axis=0)
#         if return_draws:
#             # y_like is integer draws (draws, M)
#             return mu, ppc["y_like"]
#         return mu, None

#     def simulate_season(self, future_sched,
#                         # DataFrame with ['pair_id','race_order','points'] if dynamic form
#                         hist_points=None,
#                         k_form=5,
#                         dynamic_form=True,
#                         n_mc=5000,
#                         random_state=0):
#         """
#         future_sched: rows for remaining races with covariates for self.num_features except form5
#                       (we'll set 'form5' each step if dynamic_form=True).
#                       Must include: 'race_id', 'race_order', 'pair_id', 'constructor_id', plus the other numeric regressors.
#         hist_points: if dynamic_form=True, seed last-k windows with actual historical points.
#         Returns:
#           season_summary (DataFrame per pair),
#           season_totals (array [n_pairs, total_draws]) aligned to increasing pair_id.
#         """
#         if self.model_ is None:
#             raise RuntimeError("Call fit() first.")

#         rng = np.random.default_rng(random_state)

#         # Last-k rolling windows for dynamic form
#         last_k = defaultdict(lambda: deque(maxlen=k_form))
#         if dynamic_form and hist_points is not None and not hist_points.empty:
#             for _, row in hist_points.sort_values("race_order").iterrows():
#                 last_k[int(row["pair_id"])].append(float(row["points"]))

#         # collect unique pair ids in schedule to shape outputs
#         pair_ids = np.sort(future_sched["pair_id"].astype(int).unique())
#         idx_map = {p: i for i, p in enumerate(pair_ids)}
#         total_draws = self.trace_.posterior.sizes["draw"] * \
#             self.trace_.posterior.sizes["chain"]
#         # We'll sample posterior predictive per race; to get n_mc draws, we can subset posterior draws.
#         take = min(n_mc, total_draws)
#         season_totals = np.zeros((len(pair_ids), take), dtype=float)

#         # sampling helper to slice posterior draws consistently
#         draw_idx = np.arange(total_draws)
#         rng.shuffle(draw_idx)
#         draw_idx = np.sort(draw_idx[:take])

#         with self.model_:
#             for rid, race in future_sched.sort_values("race_order").groupby("race_id"):
#                 race = race.copy().sort_values("pair_id")
#                 # build/patch form5 dynamically
#                 if dynamic_form and "form5" in self.num_features:
#                     f5 = []
#                     for p in race["pair_id"].astype(int).values:
#                         arr = np.array(last_k[p], dtype=float)
#                         f5.append(arr[-k_form:].mean() if arr.size else 0.0)
#                     race["form5"] = f5

#                 # ensure all numeric features exist
#                 for c in self.num_features:
#                     if c not in race.columns:
#                         # neutral default; better to compute upstream
#                         race[c] = 0.0

#                 Xr = self._transform(race)
#                 pair_r = self._as_int_index(race["pair_id"])
#                 cons_r = self._as_int_index(race["constructor_id"])

#                 pm.set_data({
#                     "X_data": Xr,
#                     "pair_idx": pair_r,
#                     "cons_idx": cons_r,
#                     "y_obs": np.zeros(len(race), dtype="int64")
#                 })
#                 ppc_r = pm.sample_posterior_predictive(
#                     self.trace_, var_names=["y_like"], random_seed=self.random_seed)
#                 # ppc_r["y_like"]: (total_draws, M). Subset to 'take'
#                 yr = ppc_r["y_like"][draw_idx, :]  # (take, M)

#                 # accumulate
#                 for j, p in enumerate(race["pair_id"].astype(int).values):
#                     season_totals[idx_map[p]] += yr[:, j]

#                 # update rolling windows with mean simulated outcome (fast, stable)
#                 if dynamic_form and "form5" in self.num_features:
#                     sim_mean = yr.mean(axis=0)
#                     for j, p in enumerate(race["pair_id"].astype(int).values):
#                         last_k[p].append(sim_mean[j])

#         # summarize
#         rows = []
#         for p in pair_ids:
#             s = season_totals[idx_map[p]]
#             rows.append({
#                 "pair_id": p,
#                 "future_points_mean": float(s.mean()),
#                 "future_points_p5": float(np.percentile(s, 5)),
#                 "future_points_p95": float(np.percentile(s, 95))
#             })
#         summary = pd.DataFrame(rows).sort_values(
#             "future_points_mean", ascending=False)
#         return summary, season_totals, pair_ids

#     # ---------- persistence ----------
#     def save_posterior(self, path_nc: str):
#         if self.trace_ is None:
#             raise RuntimeError("No posterior to save; fit() first.")
#         az.to_netcdf(self.trace_, path_nc)

#     def load_posterior(self, path_nc: str, model_for_set_data_needed=True):
#         # Loading the trace alone is fine for analysis, but for predictions we also need self.model_
#         # (because we use pm.set_data). Usually you'll call fit() once per season and then save.
#         self.trace_ = az.from_netcdf(path_nc)
#         if model_for_set_data_needed and self.model_ is None:
#             raise RuntimeError(
#                 "Trace loaded. To use predict/simulate, reinstantiate and refit (to rebuild the model graph).")

In [ ]:
def shrink_mean(lst, mu0, k0):
    # lst is a Python list of points
    s = np.sum(lst) if lst else 0.0
    n = len(lst)
    return (s + k0 * mu0) / (n + k0)


def build_pair_dataset(df, mu0=None, k0=2.0):
    """
    df columns (per your example):
      DriverNumber, Abbreviation, TeamName, Position, Points, Year, Race, Points_last5(list)
    Returns a frame with:
      pair_id, constructor_id, form5, points, race_order (within season), plus optional lags
    """
    df = df.copy()

    # factorize constructor + pair ids (works across full grid, not just one driver)
    df["constructor_id"] = df["TeamName"].astype(
        "category").cat.codes.astype(int)
    df["pair_key"] = df["DriverNumber"].astype(str) + "_" + df["TeamName"]
    df["pair_id"] = df["pair_key"].astype("category").cat.codes.astype(int)

    # season order
    # df["race_order"] = (df["Year"].astype(int).astype(str) + " • " + df["Race"]).astype("category").cat.codes
    df.sort_values("RaceDate", inplace=True)
    # global prior μ0: overall mean points per pair-race if not provided
    if mu0 is None:
        mu0 = float(np.nanmean(df["Points"]))

    # form5 with shrinkage toward μ0 (handles empty lists at season start)
    df["form5"] = df["Points_last5"].apply(lambda L: shrink_mean(L, mu0, k0))

    # optional: create safe lags (no leakage)
    df = df.sort_values(["pair_id", "RaceDate"]).reset_index(drop=True)
    df["points_lag1"] = df.groupby("pair_id")["Points"].shift(1).fillna(mu0)
    df["pos_lag1"] = df.groupby("pair_id")["Position"].shift(
        1).fillna(df["Position"].median())

    # final columns for the Bayesian class
    # out = df.rename(columns={"Points": "points"})
    return df

In [29]:
all_drivers_df = build_pair_dataset(season_results)

# season_results.sort_values("RaceDate")

In [66]:
all_drivers_df.head().T

,0,1,2,3,4
DriverNumber,10,10,10,10,10
BroadcastName,P GASLY,P GASLY,P GASLY,P GASLY,P GASLY
Abbreviation,GAS,GAS,GAS,GAS,GAS
DriverId,gasly,gasly,gasly,gasly,gasly
TeamName,Alpine,Alpine,Alpine,Alpine,Alpine
TeamColor,ff87bc,ff87bc,0093cc,0093cc,0093cc
TeamId,alpine,alpine,alpine,alpine,alpine
FirstName,Pierre,Pierre,Pierre,Pierre,Pierre
LastName,Gasly,Gasly,Gasly,Gasly,Gasly
FullName,Pierre Gasly,Pierre Gasly,Pierre Gasly,Pierre Gasly,Pierre Gasly


In [32]:
# all_drivers_df.head().T

In [67]:
# f1_poisson_sklearn.py
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from dataclasses import dataclass, field

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor, LogisticRegression
from sklearn.utils import resample
from joblib import dump, load

# ---------- utilities ----------


def shrink_mean(lst, mu0, k0=2.0):
    """Shrink list-mean toward mu0 with strength k0. Handles empty lists."""
    if isinstance(lst, (list, tuple, np.ndarray)) and len(lst) > 0:
        s, n = float(np.sum(lst)), len(lst)
        return (s + k0 * mu0) / (n + k0)
    return float(mu0)


def prepare_features(df: pd.DataFrame,
                     use_pos_lag=True,
                     k0=2.0,
                     prior_mu=None) -> pd.DataFrame:
    """
    Turns your raw table into model-ready rows:
      - int-coded ids
      - race_order
      - form5 from Points_last5 with shrinkage
      - safe lags (no leakage)
      - target 'points'
    """
    d = df.copy()

    # ids
    d["constructor_id"] = d["TeamName"].astype(
        "category").cat.codes.astype(int)
    d["pair_key"] = d["DriverNumber"].astype(str) + "_" + d["TeamName"]
    d["pair_id"] = d["pair_key"].astype("category").cat.codes.astype(int)

    # race order (replace with your true chronological order if you have a calendar table)
    # d["race_order"] = (d["Year"].astype(str) + " • " +
    #                    d["Race"]).astype("category").cat.codes

    # prior mean for early season
    if prior_mu is None:
        col = "points" if "points" in d.columns else "Points"
        prior_mu = float(pd.to_numeric(d[col], errors="coerce").mean())

    # form5
    if "Points_last5" in d.columns:
        d["form5"] = d["Points_last5"].apply(
            lambda L: shrink_mean(L, prior_mu, k0))
    elif "form5" in d.columns:
        d["form5"] = pd.to_numeric(
            d["form5"], errors="coerce").fillna(prior_mu)
    else:
        raise ValueError("Provide either Points_last5 or form5")

    # lags (per pair), no leakage
    d = d.sort_values(["pair_id", "RaceDate"]).reset_index(drop=True)
    d["points_lag1"] = d.groupby("pair_id")["Points"].shift(1)
    d["points_lag1"] = pd.to_numeric(
        d["points_lag1"], errors="coerce").fillna(prior_mu)

    if use_pos_lag and "Position" in d.columns:
        pos_med = pd.to_numeric(d["Position"], errors="coerce").median()
        d["pos_lag1"] = d.groupby("pair_id")["Position"].shift(1)
        d["pos_lag1"] = pd.to_numeric(
            d["pos_lag1"], errors="coerce").fillna(pos_med)

    d = d.rename(columns={"Points": "points"})
    return d

# ---------- core class ----------


@dataclass
class F1PoissonForecaster:
    num_features: list = field(default_factory=lambda: [
                               "form5", "points_lag1"])
    cat_features: list = field(default_factory=lambda: [
                               "pair_id", "constructor_id"])
    alpha: float = 1e-4                    # L2 strength for PoissonRegressor
    # if True: two-stage (logit -> Poisson)
    hurdle: bool = False
    clip_to_max_points: bool = True        # optional cap to 26 per race
    # >0 enables parameter uncertainty via bootstrapping
    bootstrap_models: int = 0
    max_iter: int = 2000
    random_state: int = 42

    def __post_init__(self):
        self._build_base_pipes()

    def _build_base_pipes(self):
        self.preproc = ColumnTransformer([
            ("num", StandardScaler(), self.num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore",
                                  ), self.cat_features)
        ])

        self.poisson = PoissonRegressor(
            alpha=self.alpha, max_iter=self.max_iter)
        self.pipe = Pipeline(
            [("pre", self.preproc), ("poisson", self.poisson)])

        if self.hurdle:
            # score vs no-score head (Bernoulli)
            self.logit = LogisticRegression(
                penalty="l2", C=1.0 / self.alpha if self.alpha > 0 else 1e6,
                max_iter=self.max_iter, solver="lbfgs"
            )
            self.pipe_zero = Pipeline(
                [("pre", self.preproc), ("logit", self.logit)])

        self.ensemble_ = []  # bootstrap models

    # -------------- fitting --------------
    def fit(self, hist: pd.DataFrame):
        X = hist[self.num_features + self.cat_features]
        y = hist["points"].astype(float).values

        # base fit
        if self.hurdle:
            y_bin = (y > 0).astype(int)
            self.pipe_zero.fit(X, y_bin)

            # fit Poisson on positive-only rows
            mask_pos = y > 0
            if mask_pos.sum() == 0:
                raise ValueError(
                    "No positive points in training for hurdle head.")
            self.pipe.fit(X.loc[mask_pos], y[mask_pos])
        else:
            self.pipe.fit(X, y)

        # optional bootstrap ensemble (parameter uncertainty)
        self.ensemble_ = []
        rng = np.random.default_rng(self.random_state)
        for b in range(self.bootstrap_models):
            idx = rng.integers(0, len(hist), size=len(hist))
            Xb, yb = X.iloc[idx], y[idx]
            if self.hurdle:
                yb_bin = (yb > 0).astype(int)
                pipe_zero_b = Pipeline([("pre", self.preproc), ("logit", LogisticRegression(
                    penalty="l2", C=1.0 / self.alpha if self.alpha > 0 else 1e6,
                    max_iter=self.max_iter, solver="lbfgs"))])
                pipe_zero_b.fit(Xb, yb_bin)

                mask_pos_b = yb > 0
                pipe_b = Pipeline([("pre", self.preproc), ("poisson", PoissonRegressor(
                    alpha=self.alpha, max_iter=self.max_iter))])
                if mask_pos_b.any():
                    pipe_b.fit(Xb.loc[mask_pos_b], yb[mask_pos_b])
                else:
                    pipe_b.fit(Xb, yb)  # fallback
                self.ensemble_.append(("hurdle", pipe_zero_b, pipe_b))
            else:
                pipe_b = Pipeline([("pre", self.preproc), ("poisson", PoissonRegressor(
                    alpha=self.alpha, max_iter=self.max_iter))])
                pipe_b.fit(Xb, yb)
                self.ensemble_.append(("plain", pipe_b))

        return self

    # -------------- helpers --------------
    def _predict_mu(self, df: pd.DataFrame) -> np.ndarray:
        X = df[self.num_features + self.cat_features]
        if self.hurdle:
            p_score = self.pipe_zero.predict_proba(X)[:, 1]
            # mean points given scoring
            mu_cond = np.clip(self.pipe.predict(X), 1e-9, None)
            mu = p_score * mu_cond
        else:
            mu = np.clip(self.pipe.predict(X), 1e-9, None)
        if self.clip_to_max_points:
            mu = np.minimum(mu, 26.0)
        return mu

    def _predict_mu_ensemble(self, df: pd.DataFrame, n_members=None) -> np.ndarray:
        """Average μ across bootstrap members + base model."""
        mus = [self._predict_mu(df)]
        if self.ensemble_:
            use = self.ensemble_ if n_members is None else self.ensemble_[
                :n_members]
            X = df[self.num_features + self.cat_features]
            for typ, *pipes in use:
                if typ == "plain":
                    mu = np.clip(pipes[0].predict(X), 1e-9, None)
                else:
                    pipe_zero_b, pipe_b = pipes
                    p_score = pipe_zero_b.predict_proba(X)[:, 1]
                    mu_cond = np.clip(pipe_b.predict(X), 1e-9, None)
                    mu = p_score * mu_cond
                mus.append(np.minimum(mu, 26.0)
                           if self.clip_to_max_points else mu)
        return np.mean(np.vstack(mus), axis=0)

    # -------------- next-race posterior via MC --------------
    def predict_next_race(self, race_df: pd.DataFrame, n_draws=10000, random_state=0):
        """
        Returns (mu_hat, draws) where:
         - mu_hat: expected points per row
         - draws: integer samples (n_rows, n_draws)
        """
        rng = np.random.default_rng(random_state)
        mu = self._predict_mu_ensemble(race_df)
        # Poisson process noise
        draws = rng.poisson(lam=np.clip(mu, 1e-9, None)
                            [:, None], size=(mu.shape[0], n_draws))
        if self.clip_to_max_points:
            draws = np.minimum(draws, 26)
        return mu, draws

    # -------------- season simulation --------------
    def simulate_season(self,
                        schedule_df: pd.DataFrame,
                        # ['pair_id','race_order','points']
                        hist_points: pd.DataFrame = None,
                        k_form=5,
                        dynamic_form=True,
                        n_draws=5000,
                        random_state=0):
        """
        schedule_df: all future pair–race rows with *known* covariates except form5 (we set it if dynamic_form).
                     Must include: pair_id, constructor_id, race_id, race_order, and any extra num_features.
        """
        rng = np.random.default_rng(random_state)

        # rolling last-k points for dynamic form
        last_k = defaultdict(lambda: deque(maxlen=k_form))
        if dynamic_form and hist_points is not None and not hist_points.empty:
            for _, row in hist_points.sort_values("race_order").iterrows():
                last_k[int(row["pair_id"])].append(float(row["points"]))

        # outputs
        pair_ids = np.sort(schedule_df["pair_id"].astype(int).unique())
        pmap = {p: i for i, p in enumerate(pair_ids)}
        season_totals = np.zeros((len(pair_ids), n_draws), dtype=float)

        # iterate race by race
        for rid, race in schedule_df.sort_values("race_order").groupby("race_id"):
            race = race.copy().sort_values("pair_id")

            # build form5 dynamically when requested
            if dynamic_form and "form5" in self.num_features:
                f5 = []
                for p in race["pair_id"].astype(int).values:
                    arr = np.array(last_k[p], dtype=float)
                    f5.append(arr[-k_form:].mean() if arr.size else 0.0)
                race["form5"] = f5

            # ensure all num_features exist
            for c in self.num_features:
                if c not in race.columns:
                    race[c] = 0.0

            mu, draws = self.predict_next_race(
                race, n_draws=n_draws, random_state=rng.integers(1, 1_000_000))

            # accumulate totals
            for j, p in enumerate(race["pair_id"].astype(int).values):
                season_totals[pmap[p]] += draws[j]

            # update rolling windows with mean simulated outcome (stable + fast)
            if dynamic_form and "form5" in self.num_features:
                sim_mean = draws.mean(axis=1)
                for j, p in enumerate(race["pair_id"].astype(int).values):
                    last_k[p].append(sim_mean[j])

        # summarize
        rows = []
        for p in pair_ids:
            s = season_totals[pmap[p]]
            rows.append({
                "pair_id": p,
                "future_points_mean": float(s.mean()),
                "future_points_p5": float(np.percentile(s, 5)),
                "future_points_p95": float(np.percentile(s, 95)),
            })
        summary = pd.DataFrame(rows).sort_values(
            "future_points_mean", ascending=False)
        return summary, season_totals, pair_ids

    # -------------- persistence --------------
    def save(self, path: str):
        dump({
            "pipe": self.pipe,
            "pipe_zero": getattr(self, "pipe_zero", None),
            "ensemble": self.ensemble_,
            "config": {
                "num_features": self.num_features,
                "cat_features": self.cat_features,
                "alpha": self.alpha,
                "hurdle": self.hurdle,
                "clip_to_max_points": self.clip_to_max_points,
                "bootstrap_models": self.bootstrap_models,
                "max_iter": self.max_iter,
                "random_state": self.random_state
            }
        }, path)

    @staticmethod
    def load(path: str):
        obj = load(path)
        cfg = obj["config"]
        f = F1PoissonForecaster(**cfg)
        f.pipe = obj["pipe"]
        f.pipe_zero = obj["pipe_zero"]
        f.ensemble_ = obj["ensemble"]
        return f

In [60]:
all_drivers_df.tail().T

,354,355,356,357,358
DriverNumber,81,81,81,81,81
BroadcastName,O PIASTRI,O PIASTRI,O PIASTRI,O PIASTRI,O PIASTRI
Abbreviation,PIA,PIA,PIA,PIA,PIA
DriverId,piastri,piastri,piastri,piastri,piastri
TeamName,McLaren,McLaren,McLaren,McLaren,McLaren
TeamColor,FF8000,FF8000,FF8000,FF8000,FF8000
TeamId,mclaren,mclaren,mclaren,mclaren,mclaren
FirstName,Oscar,Oscar,Oscar,Oscar,Oscar
LastName,Piastri,Piastri,Piastri,Piastri,Piastri
FullName,Oscar Piastri,Oscar Piastri,Oscar Piastri,Oscar Piastri,Oscar Piastri


In [76]:
# 1) Prep your historical data (your example structure works)
# converts Points_last5 -> form5, adds lags, ids
# Prepare features ONCE on the full dataset
full_df = prepare_features(all_drivers_df, k0=2.0)

# Then split by race
hist = full_df[full_df["Race"] != "Abu Dhabi Grand Prix"].copy()
next_race_df = full_df[full_df["Race"] == "Abu Dhabi Grand Prix"].copy()

# 2) Fit a plain Poisson or a hurdle model
#    Start plain; if zeros are excessive, set hurdle=True
fore = F1PoissonForecaster(
    # add "pos_lag1" if you’re okay using last race position
    num_features=["form5", "points_lag1"],
    cat_features=["pair_id", "constructor_id"],
    alpha=1e-4,
    hurdle=False,
    # optional: parameter uncertainty via bootstrapping
    bootstrap_models=50,
    random_state=123
).fit(hist)

# 3) Next-race distribution for the full grid (or a single driver)
# next_race_df must have the same feature columns; if dynamic_form, you can leave form5 blank
mu_next, draws_next = fore.predict_next_race(next_race_df, n_draws=10000)

# 4) Season simulation (static or dynamic form)
season_summary, season_totals, pair_ids = fore.simulate_season(
    # rows: one per pair–race with known covariates; include race_id, race_order
    schedule_df=future_sched_df,
    hist_points=hist[["pair_id", "race_order", "Points"]],
    k_form=5,
    dynamic_form=True,
    n_draws=5000,
    random_state=7
)

ValueError: X has 34 features, but PoissonRegressor is expecting 36 features as input.

In [77]:
next_race_df

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Year,Race,RaceDate,Points_last5,constructor_id,pair_key,pair_id,form5,points_lag1,pos_lag1
17,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 0.0, 0.0, 1.0, 0.0]",0,10_Alpine,0,1.602467,0.000000,20.0
35,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[4.0, 0.0, 1.0, 0.0, 1.0]",8,11_Red Bull Racing,1,2.316753,1.000000,10.0
53,14,F ALONSO,ALO,alonso,Aston Martin,229971,aston_martin,Fernando,Alonso,Fernando Alonso,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 8.0, 4.0, 0.0, 0.0]",1,14_Aston Martin,2,3.173896,0.000000,11.0
71,16,C LECLERC,LEC,leclerc,Ferrari,E80020,ferrari,Charles,Leclerc,Charles Leclerc,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[25.0, 18.0, 10.0, 16.0, 12.0]",2,16_Ferrari,3,13.031039,12.000000,4.0
89,18,L STROLL,STR,stroll,Aston Martin,229971,aston_martin,Lance,Stroll,Lance Stroll,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 0.0, 0.0, 0.0, 0.0]",1,18_Aston Martin,4,1.459610,0.000000,15.0
107,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[8.0, 10.0, 18.0, 8.0, 10.0]",8,1_Red Bull Racing,5,9.173896,10.000000,5.0
124,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,B6BABD,haas,Kevin,Magnussen,Kevin Magnussen,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 1.0, 0.0, 6.0, 0.0]",3,20_Haas F1 Team,6,2.459610,0.000000,12.0
142,22,Y TSUNODA,TSU,tsunoda,RB,6692FF,rb,Yuki,Tsunoda,Yuki Tsunoda,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 0.0, 0.0, 0.0, 2.0]",7,22_RB,7,1.745324,2.000000,9.0
160,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[2.0, 6.0, 0.0, 0.0, 0.0]",9,23_Williams,8,2.602467,0.000000,19.0
178,24,G ZHOU,ZHO,zhou,Kick Sauber,52e252,sauber,Guanyu,Zhou,Guanyu Zhou,...,2024,Abu Dhabi Grand Prix,2024-12-08 13:00:00,"[0.0, 0.0, 0.0, 0.0, 0.0]",4,24_Kick Sauber,9,1.459610,0.000000,13.0


In [13]:
# season_results[season_results['Abbreviation'] == 'LEC']
import arviz as az

az.summary(m.trace_, var_names=["alpha", "sigma_pair", "sigma_cons", "beta"])
az.plot_posterior(m.trace_, var_names=["beta"])

ValueError: Can only convert xarray dataarray, xarray dataset, dict, pytree (if 'dm-tree' is installed), netcdf filename, numpy array, pystan fit, emcee fit, pyro mcmc fit, numpyro mcmc fit, cmdstan fit csv filename, cmdstanpy fit to InferenceData, not NoneType

In [10]:
results_2024 = pd.read_csv(f"{PROCESSED_PATH}/2024/results.csv")
results_2025 = pd.read_csv(f"{PROCESSED_PATH}/2025/results.csv")
results_2024.head()

,Pos.,Driver,Constructor,Time/Retired,Grid,Laps,Points,race_year,race
0,1,#1Max Verstappen,Red Bull,1:54:23.566,1st,53.0,26.0,2024,japanese
1,2,#11Sergio Pérez,Red Bull,+12.535,2nd,53.0,18.0,2024,japanese
2,3,#55Carlos Sainz Jr.,Ferrari,+20.866,4th,53.0,15.0,2024,japanese
3,4,#16Charles Leclerc,Ferrari,+26.522,8th,53.0,12.0,2024,japanese
4,5,#4Lando Norris,McLaren,+29.700,3rd,53.0,10.0,2024,japanese


In [12]:
results_2024.head(25)

,Pos.,Driver,Constructor,Time/Retired,Grid,Laps,Points,race_year,race
0,1,#1Max Verstappen,Red Bull,1:54:23.566,1st,53.0,26.0,2024,japanese
1,2,#11Sergio Pérez,Red Bull,+12.535,2nd,53.0,18.0,2024,japanese
2,3,#55Carlos Sainz Jr.,Ferrari,+20.866,4th,53.0,15.0,2024,japanese
3,4,#16Charles Leclerc,Ferrari,+26.522,8th,53.0,12.0,2024,japanese
4,5,#4Lando Norris,McLaren,+29.700,3rd,53.0,10.0,2024,japanese
5,6,#14Fernando Alonso,Aston Martin,+44.272,5th,53.0,8.0,2024,japanese
6,7,#63George Russell,Mercedes,+45.951,9th,53.0,6.0,2024,japanese
7,8,#81Oscar Piastri,McLaren,+47.525,6th,53.0,4.0,2024,japanese
8,9,#44Lewis Hamilton,Mercedes,+48.626,7th,53.0,2.0,2024,japanese
9,10,#22Yuki Tsunoda,RB,+1 lap,10th,52.0,1.0,2024,japanese


In [ ]:
def rolling_points():
    return


## Feature Iteration Infrastructure

Tools for rapidly adding/removing features and measuring their impact via walk-forward evaluation.

**Workflow:**
1. Run **Multi-Year Data Loader** to build `multi_df` from the fastf1 cache
2. Toggle features in **Feature Registry** (`active: True/False`)
3. Run **Walk-Forward Eval** to see Spearman / MAE / top-3 accuracy across 2022–2024
4. Run **Feature Ablation** to see which features are load-bearing
5. Run **Experiment Log** to persist results between sessions


In [ ]:

# ── Multi-Year Data Loader ────────────────────────────────────────────────────
# Uses the fastf1 cache (ff1pkl files) — fast if already cached.
# Adds grid_position, dnf_rate5, track_form5, season_rank alongside
# the existing form5/points_lag1 features.


def load_multiyear_fastf1(fetcher, years=range(2020, 2025)):
    """Load race results for multiple seasons from the fastf1 cache."""
    dfs = []
    for year in years:
        try:
            df = fetcher.get_all_races(year)
            dfs.append(df)
            print(f"  ✓ {year}: {len(df)} driver-race rows")
        except Exception as e:
            print(f"  ⚠ {year}: {e}")
    return pd.concat(dfs, ignore_index=True)


def build_multiyear_features(season_df, k0=2.0):
    """
    Build the full feature set from a fastf1 results DataFrame.
    Expects columns: Abbreviation, TeamName, Points, GridPosition,
                     Status, Year, Race, RaceDate
    """
    df = season_df.copy()
    df["Points"] = pd.to_numeric(df["Points"], errors="coerce").fillna(0)
    df["GridPosition"] = pd.to_numeric(df["GridPosition"], errors="coerce").fillna(10)
    df = df.sort_values(["Abbreviation", "RaceDate"]).reset_index(drop=True)
    mu0 = float(df["Points"].mean())

    # ── rolling form (no leakage) ──
    df["Points_last5"] = (
        df.groupby("Abbreviation")["Points"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=0, drop=True)
    )
    df["form5"] = df["Points_last5"].apply(lambda L: shrink_mean(L, mu0, k0))
    df["points_lag1"] = df.groupby("Abbreviation")["Points"].shift(1).fillna(mu0)

    # ── grid position ──
    df["grid_position"] = df["GridPosition"]

    # ── DNF rate: classified if Finished or lapped (+N Laps) ──
    classified = (
        df["Status"].str.startswith("Finished", na=False) |
        df["Status"].str.startswith("+", na=False)
    )
    df["dnf"] = (~classified).astype(int)
    df["dnf_rate5"] = (
        df.groupby("Abbreviation")["dnf"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=0, drop=True)
        .apply(lambda L: float(np.mean(L)) if len(L) > 0 else 0.0)
    )

    # ── track-specific form ──
    track = df.sort_values(["Abbreviation", "Race", "RaceDate"]).copy()
    track["_track_last5"] = (
        track.groupby(["Abbreviation", "Race"])["Points"]
        .apply(lambda s: rolling_list(s, window=5, include_current=False))
        .reset_index(level=[0, 1], drop=True)
    )
    track["track_form5"] = track["_track_last5"].apply(
        lambda L: float(np.mean(L)) if isinstance(L, list) and len(L) > 0 else mu0
    )
    df["track_form5"] = track["track_form5"].reindex(df.index)

    # ── season rank entering race ──
    df = df.sort_values(["Year", "RaceDate", "Abbreviation"]).reset_index(drop=True)
    df["_cum"] = df.groupby(["Year", "Abbreviation"])["Points"].cumsum()
    df["_cum_lag"] = df.groupby(["Year", "Abbreviation"])["_cum"].shift(1).fillna(0)
    df["season_rank"] = (
        df.groupby(["Year", "RaceDate"])["_cum_lag"]
        .rank(ascending=False, method="min")
    )
    df.drop(columns=["_cum", "_cum_lag"], inplace=True)

    # ── categorical IDs ──
    df["constructor_id"] = df["TeamName"].astype("category").cat.codes.astype(int)
    df["pair_id"] = (df["Abbreviation"] + "_" + df["TeamName"]).astype("category").cat.codes.astype(int)

    # lowercase target for F1PoissonForecaster
    df["points"] = df["Points"]
    return df


print("Loading multi-year data from fastf1 cache...")
multi_raw = load_multiyear_fastf1(f1, years=range(2020, 2025))
multi_df = build_multiyear_features(multi_raw)
print(f"\nDataset: {multi_df.shape[0]} driver-race rows across {multi_df['Year'].nunique()} seasons")
multi_df[["Abbreviation", "Race", "Year", "form5", "grid_position", "dnf_rate5", "track_form5", "season_rank"]].head(8)


In [ ]:

# ── Feature Registry ──────────────────────────────────────────────────────────
# Toggle features on/off here — no other code needs to change.

FEATURE_CONFIG = {
    # Baseline (already implemented)
    "form5":         {"active": True,  "type": "numeric"},
    "points_lag1":   {"active": True,  "type": "numeric"},
    "pair_id":       {"active": True,  "type": "categorical"},
    "constructor_id":{"active": True,  "type": "categorical"},
    # New candidates — flip active: True to include, then rerun walk_forward_eval
    "grid_position": {"active": False, "type": "numeric"},   # qualifying/grid start
    "dnf_rate5":     {"active": False, "type": "numeric"},   # rolling DNF rate
    "track_form5":   {"active": False, "type": "numeric"},   # avg points at this circuit
    "season_rank":   {"active": False, "type": "numeric"},   # current WDC standing
}

active_numeric = [k for k, v in FEATURE_CONFIG.items() if v["active"] and v["type"] == "numeric"]
active_cat     = [k for k, v in FEATURE_CONFIG.items() if v["active"] and v["type"] == "categorical"]
print("Numeric :", active_numeric)
print("Categorical:", active_cat)


In [ ]:

# ── Walk-Forward Evaluation Harness ──────────────────────────────────────────
# Train on all seasons < test_year, evaluate on test_year.
# Use this to compare any feature set change in seconds.

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error


def top3_accuracy(df: pd.DataFrame, preds: np.ndarray) -> float:
    """Fraction of actual podium (top-3 points scorers) predicted in model's top 3."""
    df = df.copy()
    df["_pred"] = preds
    hits, total = 0, 0
    for _, race in df.groupby(["Year", "Race"]):
        actual_top3 = set(race.nlargest(3, "points").index)
        pred_top3 = set(race.nlargest(3, "_pred").index)
        hits += len(actual_top3 & pred_top3)
        total += 3
    return hits / total if total > 0 else 0.0


def walk_forward_eval(df, num_feats, cat_feats, test_seasons=(2022, 2023, 2024)):
    """Train on years < test_year, score on test_year. Returns per-season metrics."""
    rows = []
    required = num_feats + cat_feats + ["points"]
    for test_year in test_seasons:
        train = df[df["Year"] < test_year].dropna(subset=required)
        test = df[df["Year"] == test_year].dropna(subset=required)
        if train.empty or test.empty:
            continue
        model = F1PoissonForecaster(num_features=num_feats, cat_features=cat_feats)
        model.fit(train)
        preds = model._predict_mu(test)
        rows.append({
            "season": test_year,
            "spearman": round(spearmanr(test["points"], preds).statistic, 4),
            "mae": round(mean_absolute_error(test["points"], preds), 4),
            "top3_acc": round(top3_accuracy(test, preds), 4),
        })
    result = pd.DataFrame(rows)
    display(result)
    print("Mean:", result[["spearman", "mae", "top3_acc"]].mean().round(4).to_dict())
    return result


# Baseline run
baseline = walk_forward_eval(multi_df, active_numeric, active_cat)


In [ ]:

# ── Feature Ablation ──────────────────────────────────────────────────────────
# Remove one feature at a time and measure the Spearman drop.
# Run after setting FEATURE_CONFIG and loading multi_df.

baseline_scores = walk_forward_eval(multi_df, active_numeric, active_cat)
baseline_spearman = baseline_scores["spearman"].mean()
print(f"Baseline mean Spearman: {baseline_spearman:.4f}\n")

ablation = {}
for feat in active_numeric + active_cat:
    n_feats = [f for f in active_numeric if f != feat]
    c_feats = [f for f in active_cat if f != feat]
    res = walk_forward_eval(multi_df, n_feats, c_feats)
    delta = round(res["spearman"].mean() - baseline_spearman, 4)
    ablation[feat] = delta
    print(f"  -{feat}: Δ={delta:+.4f}")

ax = pd.Series(ablation).sort_values().plot(
    kind="barh", figsize=(7, 4),
    title="Spearman drop when feature removed (more negative = more important)",
    xlabel="ΔSpearman vs. baseline"
)
ax.axvline(0, color="black", linewidth=0.8)


In [ ]:

# ── Experiment Log ────────────────────────────────────────────────────────────
# Appends the current feature set's eval scores to a CSV so nothing is lost
# between sessions. Call this after any promising feature change.

import os

LOG_PATH = f"{PROCESSED_PATH}/experiment_log.csv"


def log_experiment(df, num_feats, cat_feats, note=""):
    """Run walk-forward eval and append results to the experiment log CSV."""
    run = walk_forward_eval(df, num_feats, cat_feats)
    run["features"] = str(sorted(num_feats + cat_feats))
    run["note"] = note

    if os.path.exists(LOG_PATH):
        existing = pd.read_csv(LOG_PATH)
        log = pd.concat([existing, run], ignore_index=True)
    else:
        log = run

    log.to_csv(LOG_PATH, index=False)

    summary = (
        log.groupby("features")[["spearman", "mae", "top3_acc"]]
        .mean()
        .round(4)
        .sort_values("spearman", ascending=False)
    )
    display(summary)
    return log


# Example — call with a note describing what you changed:
# log_experiment(multi_df, active_numeric, active_cat, note="baseline: form5 + points_lag1")
